# Ambr15 fed-batch data: assembly of the analysis table

Three sources are combined into one long table, with one row per vessel × parameter × time point:

| source | file | content |
|---|---|---|
| design | `data/Target and Random_DoE design.xlsx` | process condition of every vessel; random-integration clone per vessel |
| ambr15 + Bioprofile FLEX2 | `data/Ambr_Bioprofile/*.csv` (one file per ambr run) | daily metabolites, viable cell density, viability |
| Octet | `data/Octet/tdo_new.xlsx` + B4 plate reports | product titer (µg/mL) per vessel and day, with dilution factors |

In [11]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("data")
PARAMETERS = ["CELL_COUNT", "CELL_VIABILITY", "LACTATE", "AMMONIUM", "GLUCOSE", "GLUTAMINE", "GLUTAMATE", "CALCIUM", "POTASSIUM", "SODIUM", "FLEX_PH"]

## 1. Design

RMCE clones were each cultivated in one ambr15 run covering the full factorial of 24 conditions
(temperature shift on day 4 × feed × basal medium × pH set point). The random-integration (RI) clones of
each product shared one run: 8 of the 24 conditions × 3 clones. A condition is identified by its RMCE
DOE name (N1 to N24), so an RI vessel carries the identifier of the RMCE condition it replicates.

In [12]:
def doe_to_vessel(name):
    """DOE experiment name N1..N24 -> ambr vessel id CS1_1..CS1_12, CS2_1..CS2_12."""
    n = int(str(name).lstrip("N"))
    assert 1 <= n <= 24
    return f"CS{1 if n <= 12 else 2}_{n if n <= 12 else n - 12}"

design_file = DATA / "Target and Random_DoE design.xlsx"
design_rmce = pd.read_excel(design_file, skiprows=3, usecols="B:F", engine='calamine').rename(columns={
    "DOE Experiment Name": "vessel_doe", "Temperature shift Day 4": "temp_shift",
    "Feed": "feed", "Medium": "medium", "pH": "ph"})
design_rmce["condition"] = design_rmce["vessel_doe"]
design_ri = pd.read_excel(design_file, skiprows=3, usecols="L:S", engine='calamine').rename(columns={
    "Target integration condition": "condition", "DOE Experiment Name.1": "vessel_doe",
    "Temperature shift Day 4.1": "temp_shift", "Feed.1": "feed", "Medium.1": "medium", "pH.1": "ph",
    "enb_random_clone": "ETN", "epo_random_clone": "EPO"}).astype({"ETN": str, "EPO": str})
for design in (design_rmce, design_ri):
    design["vid"] = design["vessel_doe"].map(doe_to_vessel)

factors = ["temp_shift", "feed", "medium", "ph"]
assert len(design_rmce) == 24 and len(design_rmce[factors].drop_duplicates()) == 24
check = design_ri.merge(design_rmce[["condition"] + factors], on="condition", suffixes=("", "_rmce"))
assert len(check) == 24 and all((check[c] == check[c + "_rmce"]).all() for c in factors)
for product in ("ETN", "EPO"):
    assert (design_ri.groupby(product).size() == 8).all()
    assert design_ri.groupby([product, "condition"]).size().max() == 1

design_rmce

,vessel_doe,temp_shift,feed,medium,ph,condition,vid
0,N1,34,Cell Boost 5,FortiCHO,7.1,N1,CS1_1
1,N2,34,Cell Boost 5,FortiCHO,6.9,N2,CS1_2
2,N3,34,Cell Boost 5,FortiCHO,7.3,N3,CS1_3
3,N4,34,Cell Boost 5,OptiCHO,7.1,N4,CS1_4
4,N5,34,Cell Boost 5,OptiCHO,6.9,N5,CS1_5
5,N6,34,Cell Boost 5,OptiCHO,7.3,N6,CS1_6
6,N7,34,Cell Boost 7,FortiCHO,7.1,N7,CS1_7
7,N8,34,Cell Boost 7,FortiCHO,6.9,N8,CS1_8
8,N9,34,Cell Boost 7,FortiCHO,7.3,N9,CS1_9
9,N10,34,Cell Boost 7,OptiCHO,7.1,N10,CS1_10


In [13]:
design_ri

,condition,vessel_doe,temp_shift,feed,medium,ph,ETN,EPO,vid
0,N3,N1,34,Cell Boost 5,FortiCHO,7.3,3,4,CS1_1
1,N4,N2,34,Cell Boost 5,OptiCHO,7.1,3,4,CS1_2
2,N5,N3,34,Cell Boost 5,OptiCHO,6.9,3,4,CS1_3
3,N12,N4,34,Cell Boost 7,OptiCHO,7.3,3,4,CS1_4
4,N3,N5,34,Cell Boost 5,FortiCHO,7.3,24,12,CS1_5
5,N4,N6,34,Cell Boost 5,OptiCHO,7.1,24,12,CS1_6
6,N5,N7,34,Cell Boost 5,OptiCHO,6.9,24,12,CS1_7
7,N12,N8,34,Cell Boost 7,OptiCHO,7.3,24,12,CS1_8
8,N3,N9,34,Cell Boost 5,FortiCHO,7.3,26,14,CS1_9
9,N4,N10,34,Cell Boost 5,OptiCHO,7.1,26,14,CS1_10


## 2. Ambr15 and Bioprofile FLEX2

In [14]:
frames, runs = [], []
for csv_file in sorted((DATA / "Ambr_Bioprofile").glob("*.csv")):
    raw = pd.read_csv(csv_file, low_memory=False)
    timestamps = pd.to_datetime(raw["Date Time"].str.replace(".", ":").str.replace("Sept", "Sep"), format="%d/%b/%Y %H:%M")
    product = "ETN" if "ENB" in csv_file.name else "EPO"
    integration = "RI" if csv_file.name.startswith("RANDOM") else "RMCE"
    clone = None if integration == "RI" else csv_file.name.split("_")[0]
    # the A10 export begins on culture day 4
    seconds = ((timestamps - timestamps.min()).dt.total_seconds() + (4 * 86400 if clone == "A10" else 0)).astype("int64")
    long = (raw[[c for c in raw.columns if c != "Date Time" and c.split("-", 2)[-1].strip() in PARAMETERS]]
            .apply(pd.to_numeric, errors="coerce").set_axis(seconds).stack().dropna()
            .rename_axis(["seconds", "col"]).reset_index(name="value"))
    parts = long.pop("col").str.split("-", n=2, expand=True)
    long["vid"] = parts[0] + "_" + parts[1]
    long["parameter"] = parts[2].str.strip()
    long["product"] = product
    long["integration"] = integration
    long["clone"] = long["vid"].map(design_ri.set_index("vid")[product]) if clone is None else clone
    long["run"] = csv_file.stem
    assert long["clone"].notna().all()
    frames.append(long)
    runs.append(dict(run=csv_file.stem, product=product, integration=integration, clones=", ".join(sorted(long.clone.unique())), start=timestamps.min().date(), end=timestamps.max().date()))
flex = pd.concat(frames, ignore_index=True)
# A10 FLEX2 samples before day 5 are not valid
flex = flex[~((flex.clone == "A10") & (flex.seconds < 5 * 86400))].assign(note="")
assert set(flex.parameter) == set(PARAMETERS)

flex

,seconds,value,vid,parameter,product,integration,clone,run,note
264,483300,6.47,CS1_1,CELL_COUNT,ETN,RMCE,A10,A10_ENB,
265,483300,98.30,CS1_1,CELL_VIABILITY,ETN,RMCE,A10,A10_ENB,
266,483300,3.98,CS1_1,LACTATE,ETN,RMCE,A10,A10_ENB,
267,483300,5.98,CS1_1,GLUCOSE,ETN,RMCE,A10,A10_ENB,
268,483300,5.75,CS1_1,GLUTAMATE,ETN,RMCE,A10,A10_ENB,
...,...,...,...,...,...,...,...,...,...
30467,1265520,96.60,CS2_10,CELL_VIABILITY,EPO,RI,14,RANDOM_EPO,
30468,1265940,6.75,CS2_11,CELL_COUNT,EPO,RI,14,RANDOM_EPO,
30469,1265940,99.00,CS2_11,CELL_VIABILITY,EPO,RI,14,RANDOM_EPO,
30470,1266360,4.55,CS2_12,CELL_COUNT,EPO,RI,14,RANDOM_EPO,


## 3. Octet titer

Concentration = dilution factor × Octet reading (µg/mL = mg/L). 
Readings handled differently:

* Readings below the quantification range are set to 0.
* For 4 day 14 samples without a valid reading, we take the last good value.
* Readings above the range on other days are excluded.

In [15]:
epo, etn = pd.read_excel(DATA / "Octet/tdo_new.xlsx", sheet_name=["EPO", "ENB"], engine='calamine').values()
for m in (epo, etn):
    m["Clone"] = m["Clone"].astype(str)
    m["conc"] = m["Dilution Factor"] * pd.to_numeric(m["old_raw"], errors="coerce")
    m["note"] = ""
    assert not m.duplicated(["Clone", "Sample ID", "Day"]).any()

titer_notes = []

def carry_forward(m, clone, vessel, day, from_day, reason):
    tgt = (m.Clone == clone) & (m["Sample ID"] == vessel) & (m.Day == day)
    src = (m.Clone == clone) & (m["Sample ID"] == vessel) & (m.Day == from_day)
    assert tgt.sum() == 1 and src.sum() == 1
    m.loc[tgt, "conc"] = m.loc[src, "conc"].to_numpy()
    m.loc[tgt, "note"] = f"day {from_day} value carried forward ({reason})"
    titer_notes.append(dict(product="ETN", clone=clone, vessel=vessel, day=day,
                            reading=m.loc[tgt, "old_raw"].iloc[0], value=float(m.loc[tgt, "conc"].iloc[0]),
                            handling=m.loc[tgt, "note"].iloc[0]))

carry_forward(etn, "3", 1, 14, 13, ">5-fold jump from day 13")
carry_forward(etn, "A10", 6, 14, 13, "below range")
carry_forward(etn, "3", 16, 14, 12, "above range on days 13 and 14")
carry_forward(etn, "24", 20, 14, 12, "above range on days 13 and 14")

too_low = epo["old_raw"].astype(str) == "Too Low"
epo.loc[too_low, "conc"] = 0.0
epo.loc[too_low, "note"] = "below quantification range, set to 0"
titer_notes += [dict(product="EPO", clone=r.Clone, vessel=r["Sample ID"], day=r.Day, reading="Too Low", value=0.0, handling=r.note) for _, r in epo[too_low].iterrows()]
titer_excluded = pd.concat([m.loc[m.conc.isna(), ["Clone", "Sample ID", "Day", "old_raw"]].assign(product=p) for p, m in (("EPO", epo), ("ETN", etn))], ignore_index=True)
epo, etn = epo[epo.conc.notna()].copy(), etn[etn.conc.notna()].copy()

# each plate carried the A10 vessel-12 days 4-14 as bridge samples. 
# B4 values are mapped to the A10 reference scale by piecewise-linear
# interpolation through the bridge pairs

OUTSIDE_BRIDGE = "proportional"
ref = etn.loc[(etn.Clone == "A10") & (etn["Sample ID"] == 12), "conc"].astype(float).to_numpy()
assert len(ref) == 10
y = np.delete(ref, 2)[::-1]
b4_bridges = []
plates = sorted((DATA / "Octet").glob("*B4*"))
assert len(plates) == 4
for plate_file in plates:
    od = pd.read_excel(plate_file, sheet_name="Results", engine='calamine')
    od = od[od["Sample ID"].astype(str).str.contains("A10")]
    od = od.sort_values(by="Sample", key=lambda c: c.str.extract(r"(\d+)$").astype(int)[0])
    assert len(od) == 10
    x = np.delete(od["Calc Conc."].astype(float).to_numpy(), 2)[::-1]
    sel = etn["Folder name"].str.lower() == f"{plate_file.stem.lower()}.xls"
    assert sel.sum() > 0 and (etn.loc[sel, "Clone"] == "B4").all()
    before = etn.loc[sel, "conc"].astype(float).to_numpy()
    mapped = np.interp(before, x, y)
    mapped = np.where(before < x[0], before * y[0] / x[0], mapped)
    mapped = np.where(before > x[-1], before * y[-1] / x[-1], mapped)
    etn.loc[sel, "conc"] = mapped
    outside = sel.copy()
    outside[sel] = (before > x.max()) | (before < x.min())
    etn.loc[outside, "note"] = "B4 plate bridge scaled"
    b4_bridges.append(dict(plate=plate_file.name.split("Vessel ")[1].split("_")[0].strip(), x=x, y=y, n=int(sel.sum()), n_outside=int(outside.sum())))
assert (etn.Clone == "B4").sum() == sum(b["n"] for b in b4_bridges)

titer = pd.concat([pd.DataFrame({
    "seconds": (m["Day"] * 86400).astype("int64"), "value": m["conc"].astype(float),
    "vid": m["Sample ID"].map(doe_to_vessel), "parameter": "titer", "product": p, "clone": m["Clone"],
    "integration": np.where(m["Clone"].str.isdigit(), "RI", "RMCE"), "note": m["note"]})
    for p, m in (("EPO", epo), ("ETN", etn))], ignore_index=True)

titer

,seconds,value,vid,parameter,product,clone,integration,note
0,1123200,492.500000,CS2_3,titer,EPO,4,RI,
1,1209600,490.000000,CS2_3,titer,EPO,4,RI,
2,950400,437.500000,CS2_3,titer,EPO,4,RI,
3,1209600,432.500000,CS2_4,titer,EPO,4,RI,
4,691200,360.000000,CS2_3,titer,EPO,4,RI,
...,...,...,...,...,...,...,...,...
1407,345600,2.957000,CS2_10,titer,ETN,26,RI,
1408,604800,5.446791,CS2_1,titer,ETN,A5,RMCE,
1409,518400,4.247134,CS2_1,titer,ETN,A5,RMCE,
1410,345600,1.745495,CS2_1,titer,ETN,A5,RMCE,


In [16]:
cond_cols = ["condition", "temp_shift", "feed", "medium", "ph"]
conditions = pd.concat([design_rmce.assign(integration="RMCE"), design_ri.assign(integration="RI")]).set_index(["integration", "vid"])[cond_cols]
ambr = (pd.concat([flex.drop(columns="run"), titer], ignore_index=True)
        .join(conditions, on=["integration", "vid"])
        .merge(flex[["product", "integration", "clone", "run"]].drop_duplicates(), on=["product", "integration", "clone"], how="left"))
assert ambr[cond_cols + ["run"]].notna().all().all()

# vessel CS2_1 of clone A5 (condition N13) failed during cultivation and is excluded
ambr = ambr[~((ambr.clone == "A5") & (ambr.vid == "CS2_1"))].copy()

ambr["day"] = ambr["seconds"] / 86400
ambr["vessel"] = ambr["product"] + "_" + ambr["clone"] + "_" + ambr["vid"]
string_cols = ["product", "integration", "clone", "run", "vid", "vessel", "condition", "feed", "medium", "parameter", "note"]
ambr = (ambr[["product", "integration", "clone", "run", "vid", "vessel", "condition", "temp_shift", "feed",
              "medium", "ph", "parameter", "seconds", "day", "value", "note"]]
        .astype({**dict.fromkeys(string_cols, "string"), "temp_shift": "int64", "ph": float})
        .sort_values(["product", "integration", "clone", "vid", "parameter", "seconds"]).reset_index(drop=True))
assert not ambr.duplicated(["vessel", "parameter", "seconds"]).any()
assert ambr.value.notna().all()

vessel_counts = ambr.drop_duplicates("vessel").groupby(["integration", "product", "clone"]).agg(
    vessels=("vessel", "size"), conditions=("condition", "nunique")).reset_index()
assert set(vessel_counts.query("integration == 'RI'").vessels) == {8}
assert set(vessel_counts.query("integration == 'RMCE'").vessels) <= {23, 24}

In [17]:
vessel_counts

,integration,product,clone,vessels,conditions
0,RI,EPO,12,8,8
1,RI,EPO,14,8,8
2,RI,EPO,4,8,8
3,RI,ETN,24,8,8
4,RI,ETN,26,8,8
5,RI,ETN,3,8,8
6,RMCE,EPO,A4,24,24
7,RMCE,EPO,A8,24,24
8,RMCE,EPO,B1,24,24
9,RMCE,ETN,A10,24,24


In [18]:
ambr

,product,integration,clone,run,vid,vessel,condition,temp_shift,feed,medium,ph,parameter,seconds,day,value,note
0,EPO,RI,12,RANDOM_EPO,CS1_5,EPO_12_CS1_5,N3,34,Cell Boost 5,FortiCHO,7.3,AMMONIUM,50400,0.583333,0.940000,
1,EPO,RI,12,RANDOM_EPO,CS1_5,EPO_12_CS1_5,N3,34,Cell Boost 5,FortiCHO,7.3,AMMONIUM,77640,0.898611,1.120000,
2,EPO,RI,12,RANDOM_EPO,CS1_5,EPO_12_CS1_5,N3,34,Cell Boost 5,FortiCHO,7.3,AMMONIUM,147960,1.712500,2.010000,
3,EPO,RI,12,RANDOM_EPO,CS1_5,EPO_12_CS1_5,N3,34,Cell Boost 5,FortiCHO,7.3,AMMONIUM,225060,2.604861,3.440000,
4,EPO,RI,12,RANDOM_EPO,CS1_5,EPO_12_CS1_5,N3,34,Cell Boost 5,FortiCHO,7.3,AMMONIUM,308040,3.565278,5.440000,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31529,ETN,RMCE,B4,B4_ENB,CS2_9,ETN_B4_CS2_9,N21,31,Cell Boost 7,FortiCHO,7.3,titer,864000,10.000000,759.224254,
31530,ETN,RMCE,B4,B4_ENB,CS2_9,ETN_B4_CS2_9,N21,31,Cell Boost 7,FortiCHO,7.3,titer,950400,11.000000,852.778302,
31531,ETN,RMCE,B4,B4_ENB,CS2_9,ETN_B4_CS2_9,N21,31,Cell Boost 7,FortiCHO,7.3,titer,1036800,12.000000,962.323557,
31532,ETN,RMCE,B4,B4_ENB,CS2_9,ETN_B4_CS2_9,N21,31,Cell Boost 7,FortiCHO,7.3,titer,1123200,13.000000,1031.620062,


In [19]:
ambr.to_feather("ambr_long.feather")